# Training, Validation & Test Sets (6/19/26)

---

*Codecademy — MLE Path: Supervised Learning I. Concise grad-student notes.*

## The core problem: don't grade yourself on your own homework

- A model that memorizes its training data can score perfectly on it and still be **useless on new data** (overfitting).
- So we must evaluate on data the model has **never seen** during training. That means *partitioning* the dataset before doing anything.
- **Generalization** = performance on unseen data. That's the only thing we actually care about.

## The three splits

| Set | Purpose | Touches model params? | Touches your choices? |
|---|---|---|---|
| **Training** | Model *learns* parameters (weights, splits, etc.) | ✅ fits on it | — |
| **Validation** | Tune **hyperparameters** & compare models; the "dev" set | ❌ | ✅ you pick based on it |
| **Test** | Final, one-shot estimate of generalization | ❌ | ❌ (look once, at the end) |

- **Typical ratios:** `80 / 20` (train/test) when no validation set, or `60 / 20 / 20` (train/val/test). `70/15/15` also common. Bigger data → you can afford a smaller % for val/test.
- **Hyperparameter** = a setting *you* choose before training (k in KNN, tree depth, learning rate), as opposed to a parameter the model learns.

### Why validation AND test? (the subtle part)

- You tune hyperparameters by checking performance on the **validation** set repeatedly.
- Over many rounds of tuning, you start **fitting your choices to the validation set** — it leaks into the model indirectly. Its score becomes optimistic.
- The **test** set stays in a vault, untouched until the very end, to give an *honest* final number. Touch it once.

## Two-way split with scikit-learn

`train_test_split` is the workhorse. `random_state` makes the shuffle reproducible; `test_size` is the held-out fraction.

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split

# toy data: 100 samples, 3 features
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 3))
y = rng.integers(0, 2, size=100)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("train:", X_train.shape, "  test:", X_test.shape)   # 80 / 20


train: (80, 3)   test: (20, 3)


## Three-way split (train / validation / test)

`train_test_split` only cuts in two, so call it **twice**: first carve off test, then carve validation out of what's left.

In [2]:
# 1) hold out 20% as TEST
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 2) from the remaining 80%, take 25% as VALIDATION -> 0.25 * 0.80 = 0.20 of total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print("train:", len(X_train), " val:", len(X_val), " test:", len(X_test))  # 60 / 20 / 20


train: 60  val: 20  test: 20


> **Watch the fractions:** the second split's `test_size` is a fraction *of the remaining data*, not of the original. To get 20% of the whole from the leftover 80%, you ask for `0.20 / 0.80 = 0.25`.

## The problem with a single validation set → N-Fold Cross-Validation

- A single train/val split is **noisy**: your score depends on *which* points happened to land in validation. Unlucky split → misleading number. Wasteful too — val data never trains the model.
- **N-Fold (k-fold) Cross-Validation** fixes both:
  1. Split training data into **N equal folds**.
  2. Train on **N−1** folds, validate on the held-out 1 fold.
  3. Rotate so **every fold is the validation set exactly once** → N scores.
  4. **Average** the N scores for a far more stable estimate.
- Every point gets used for both training and validation (just never simultaneously). Common choice: **N = 5 or 10**.
- **Cost:** you train the model N times. (The test set still stays in the vault — CV happens *within* the non-test data.)

In [3]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=kf, scoring="accuracy")
print("per-fold accuracy:", np.round(scores, 3))
print("mean:", round(scores.mean(), 3), " std:", round(scores.std(), 3))
# Report mean ± std -- the std tells you how sensitive the estimate is to the split.


per-fold accuracy: [0.417 0.5   0.333 0.25  0.333]
mean: 0.367  std: 0.085


## Honest end-to-end workflow

1. **Split off the test set first** and lock it away.
2. On the remaining data, use **cross-validation** to tune hyperparameters / pick the model.
3. **Refit** the chosen model on *all* non-test data.
4. Evaluate **once** on the test set → that's your reported generalization performance.

> Anything that lets test-set info influence training = **data leakage** = optimistic, dishonest scores. (Same rule as fitting a scaler — see `Normalization_Lesson.ipynb`.)

## TL;DR

- **Train** fits params · **Validation** tunes hyperparameters · **Test** = final honest score, used once.
- Ratios: `80/20` or `60/20/20` (scale val/test down as data grows).
- `train_test_split` cuts in two → call it twice for a three-way split; mind the fraction-of-remainder gotcha.
- **N-fold CV** averages over N rotating validation folds → stable estimate, full data usage, at N× training cost.
- Never let test data inform training.